# 11e — Jeux globaux : la coordination avec information incomplète
**Version jumelle** : [C#](GameTheory-11e-Global-Games-CSharp.ipynb)
**Navigation** : [<< 11-BayesianGames](GameTheory-11-BayesianGames-Python.ipynb) | [11c-Global-Games (sibling)](GameTheory-11c-Global-Games-Python.ipynb) | [Index](README.md)


**Autour de 11** — prolonge [11 · Jeux bayésiens](GameTheory-11-BayesianGames-Python.ipynb), [04 · Équilibres de Nash](GameTheory-04-NashEquilibrium-Python.ipynb) et le jumeau [11c — sélection d'équilibre par trembles](GameTheory-11c-Global-Games-Python.ipynb) (mêmes jeux globaux, autre question : ici l’étude du **plus petit seuil fini mesuré**, là le modèle de trembles).

Un jeu de coordination 2×2 à information **complète** porte **deux** équilibres de Nash purs (les deux accords) plus un équilibre mixte intérieur : la **multiplicité** n'est pas tranchée. Le dispositif des **jeux globaux** (Carlsson & van Damme 1993 ; Morris & Shin 1998) donne à chaque joueur un **signal bruité** d'un fondamental commun `θ`.

**La question de ce carnet** : quel **plus petit seuil fini** la recherche mesure-t-elle, et vers quel équilibre ce seuil conduit-il quand le bruit disparaît ? Le plus petit seuil fini est défini relativement à une fenêtre de recherche ; les mesures de ce carnet suggèrent une limite vers `C` lorsque le bruit décroît. Plusieurs racines de `g_sym` peuvent toutefois coexister : le carnet les compte sans conclure à une sélection globale.

> Cœur du carnet : intégration gaussienne en forme close + recherche de racine, avec une fenêtre adaptée au bruit. Le résultat est **falsifiable**.

La lecture suit une trajectoire en cinq gestes : compter les équilibres du jeu complet (section 1), regarder le canal dégradé qu'est le signal (section 2), calculer le plus petit seuil et observer sa proximité à `C` (section 3), lire les probabilités associées à ce seuil autour de `θ = C` (section 3 toujours) — et relier le tout (section 4). Les trois grandeurs affichées en route — `p*`, `k*`, P(les deux jouent H) — sont les coordonnées du même récit : le calcul compare des racines dans une classe de seuils.

In [1]:
import numpy as np
import math

# --- Loi normale standard, via la stdlib (pas besoin de scipy) ---
def phi(x):
    """Densité normale standard : phi(x) = exp(-x^2/2) / sqrt(2 pi)."""
    return math.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)

def Phi(x):
    """Fonction de répartition normale standard : Phi(x) = 0.5 (1 + erf(x/sqrt 2))."""
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

# --- Paramètres du jeu (partagés par tout le carnet) ---
TAU2 = 1.0   # variance a priori du fondamental theta ~ N(0, TAU2)
C    = 1.0   # coût d'agir seul : payer H face a L rapporte -C

np.random.seed(42)   # reproductibilité des tirages (si simulation)
print("Imports OK  |  phi(0) =", round(phi(0), 4), " |  Phi(0) =", Phi(0), " |  C =", C)

Imports OK  |  phi(0) = 0.3989  |  Phi(0) = 0.5  |  C = 1.0


### Lecture du résultat — la forme close, sans `scipy`

Deux fonctions portent tout le calcul du carnet : `phi` (densité) et `Phi` (fonction de répartition) de la normale standard, **implémentées à la main** sur la stdlib (`math.exp`, `math.erf`) — pas de dépendance `scipy`. La ligne de contrôle le confirme : `phi(0) = 0.3989` (le pic de la cloche, `1/√(2π)`) et `Phi(0) = 0.5` (la symétrie : la moitié de la masse est à gauche de 0). Ces deux valeurs sont les **deux** points que toute implémentation correcte doit restituer ; c'est le contrôle de cohérence du reste du carnet, où l'utilité espérée de `H` s'écrit **en forme close** à partir de `phi` et `Phi` (section 3).

## 1. Le jeu de coordination complet — la multiplicité d'origine

Deux joueurs, chacun joue **H** (investir / entrer) ou **L** (s'abstenir). Il existe un **fondamental** `θ` (par ex. la rentabilité d'un projet) : jouer H rapporte `θ` **si l'autre joue aussi H**, mais coûte `C` si l'on joue H **seul**.

| Moi \ Adverse | **H** | **L** |
|---|---|---|
| **H** | `θ` | `−C` |
| **L** | `0` | `0` |

À **information complète** (chaque joueur connaît `θ`) :
- **(H,H)** est un équilibre de Nash **si `θ ≥ 0`** (H est une meilleure réponse faible à H à l'égalité) ;
- **(L,L)** est **toujours** un équilibre (face à L, L bat H : `0 > −C`) ;
- l'**équilibre mixte** (chacun joue H avec probabilité `p* = C/(θ+C)`) existe quand `θ > 0`.

**Donc, pour `θ > 0` : deux équilibres purs + un mixte intérieur = multiplicité.** Le jeu seul ne dit pas lequel se réalisera.

Le mixte mérite un détour : il sépare deux régimes de meilleure réponse. À `θ = +0.5`, `p* = 0.6667` est la fréquence qui rend l'adversaire **indifférent** — un joueur qui mixte exactement à `p*` face au même stratège gagne zéro en espérance, comme s'il jouait L. Baissez légèrement votre probabilité : l'autre a intérêt à jouer L tout court. Augmentez-la : H domine. Le mixte sépare donc les réponses pures à une probabilité adverse donnée. Le calcul suivant cherche les seuils finis compatibles avec les signaux bruités ; il ne simule pas une dynamique d'attraction.

In [2]:
# === Section 1 : le jeu de coordination complet (information complète) ===

def equilibres_purs(theta):
    """Liste des équilibres de Nash purs du jeu 2x2 de coordination."""
    ne = []
    if theta >= 0:            # (H,H) : H meilleure réponse a H  <=>  theta >= 0
        ne.append(("H", "H"))
    ne.append(("L", "L"))    # (L,L) : toujours un équilibre (0 > -C)
    return ne

def equilibre_mixte(theta):
    """Proba p* d'indifférence (mixte) ; None si hors de [0, 1]."""
    # Indifférence :  p* (theta) + (1 - p*) (-C) = 0   =>   p* = C / (theta + C)
    p = C / (theta + C)
    return p if 0.0 <= p <= 1.0 else None

for theta in (0.5, -0.5):
    print(f"\ntheta = {theta:+.1f}")
    print("  Nash purs   :", equilibres_purs(theta))
    pm = equilibre_mixte(theta)
    print("  Nash mixte p* :", (round(pm, 4) if pm is not None else "aucun (hors [0,1])"))
print("\n-> Pour theta > 0 : multiplicité (2 purs + 1 mixte). Pour theta < 0 : (L,L) seul ; à theta=0, (H,H) est faible.")


theta = +0.5
  Nash purs   : [('H', 'H'), ('L', 'L')]
  Nash mixte p* : 0.6667

theta = -0.5
  Nash purs   : [('L', 'L')]
  Nash mixte p* : aucun (hors [0,1])

-> Pour theta > 0 : multiplicité (2 purs + 1 mixte). Pour theta < 0 : (L,L) seul ; à theta=0, (H,H) est faible.


### Lecture — la multiplicité en nombres

La sortie sépare deux mondes en un seul paramètre. À `θ = +0.5`, le jeu porte **deux** équilibres purs, `('H', 'H')` et `('L', 'L')`, plus un mixte `p* = 0.6667` : chacun investit avec deux chances sur trois, et c'est exactement la fréquence qui rend l'autre indifférent — à `p*`, jouer H et jouer L rapportent tous deux zéro en espérance. À `θ = −0.5`, la liste ne contient plus que `('L', 'L')` : le gain commun d'un investissement réussi est négatif, H n'est jamais une meilleure réponse, et le mixte disparaît car `p* = C/(θ+C)` déborderait de `[0, 1]` — la fonction le signale en renvoyant `None`.

Retenez la mécanique du mixte : l'indifférence exige `p·θ − (1 − p)·C = 0`, d'où `p* = C/(θ+C)`. Quand `θ` tend vers 0 par valeurs positives, `p*` tend vers 1 — il faut être presque sûr que l'autre investit pour oser investir dans un projet à peine rentable. C'est cette fragilité de l'accord ('H', 'H'), qui ne survit que par certitude mutuelle, que le bruit va exploiter.

## 2. Le signal bruité et la stratégie à seuil

Personne ne connaît `θ` directement. Chaque joueur `i` observe un **signal bruité**

$$s_i = \theta + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0, \sigma^2) \ \text{(indépendants).}$$

On restreint aux **stratégies à seuil** : jouer **H** si et seulement si `s_i ≥ k`. La décision se résume à une **frontière** `k` dans l'espace des signaux — la classe de stratégies naturelle quand l'information est un signal du fondamental.

Économiquement, `s = θ + ε` se lit : chaque joueur observe le fondamental à travers son propre canal dégradé — deux analystes voient la même économie avec des modèles et des données légèrement différents. La variance `σ²` mesure la qualité du canal ; restreindre aux stratégies à seuil dit seulement que la décision finale est binaire : au-dessus de `k`, le signal paraît assez bon pour tenter l'investissement. Toute la subtilité du jeu passera dans le choix de cette frontière.

In [3]:
# === Section 2 : le signal bruité + stratégie à seuil ===

def signaux(theta, sigma, n=1, rng=None):
    """n signaux bruités d'un même fondamental : s = theta + N(0, sigma^2)."""
    rng = rng or np.random.default_rng()
    return theta + rng.normal(0.0, sigma, size=n)

theta0, sigma0 = 0.5, 0.3
ex = signaux(theta0, sigma0, n=6, rng=np.random.default_rng(42))
print(f"theta = {theta0},  sigma = {sigma0}")
print("6 signaux tirés :", np.round(ex, 3))
print("-> chaque joueur voit un theta légèrement décalé ; la décision est une frontière (seuil k) dans cet espace de signaux.")

theta = 0.5,  sigma = 0.3
6 signaux tirés : [ 0.591  0.188  0.725  0.782 -0.085  0.109]
-> chaque joueur voit un theta légèrement décalé ; la décision est une frontière (seuil k) dans cet espace de signaux.


### Lecture — ce que chaque joueur voit

Pour le même fondamental `θ=0,5` et `σ=0,3`, les six signaux de cette exécution s'étalent de `−0,085` à `0,782` : un signal est négatif malgré le fondamental positif, tandis que le signal `0,782` lui est nettement supérieur. Le tirage varie avec le générateur ; les signaux sont indépendants conditionnellement à `θ`.

Le canal bruité donne ainsi des observations différentes d'une même économie. Chaque joueur raisonne sur le signal de l'autre sans le connaître : le modèle combine cette incertitude avec la mise à jour gaussienne du fondamental. Le seuil `k` est la frontière au-dessus de laquelle jouer H. La question suivante cherche les seuils compatibles avec ces croyances, puis distingue la plus petite racine finie mesurée des autres racines.


## 3. Le plus petit seuil fini mesuré

Avec un **a priori** `θ ~ N(0, τ²)` et un signal gaussien, la croyance a posteriori d'un joueur de signal `s` est gaussienne : `θ | s ~ N(m, v)` avec

$$m = \tfrac{v\,s}{\sigma^2}, \qquad v = \Big(\tfrac{1}{\tau^2} + \tfrac{1}{\sigma^2}\Big)^{-1}.$$

Face à un adversaire de seuil `k`, l'utilité espérée de **H** s'écrit **en forme close** (intégration gaussienne) :

$$\mathbb{E}[U_H \mid s] = (m+C)\,\Phi\!\Big(\tfrac{m-k}{w}\Big) + \tfrac{v}{w}\,\varphi\!\Big(\tfrac{m-k}{w}\Big) - C, \qquad w = \sqrt{v + \sigma^2},$$

où `Φ` et `φ` sont la fonction de répartition et la densité de la normale standard. Un **seuil d'équilibre** `k*` est une valeur où un joueur de signal `s = k*` est **indifférent** face à un adversaire de même seuil : c'est une racine de `g(k) = U_H(k; k) = 0`. Le **plus petit seuil fini trouvé dans la fenêtre** est étudié ici ; les autres racines et les équilibres hors de cette classe ne sont pas écartés.

**Le point clé** : dans les fenêtres et bruits étudiés, la plus petite racine finie trouvée se rapproche du **coût d'agir seul** `C`. Cela ne prouve **pas** qu'il est la seule racine : la mesure suivante porte sur les racines de la fenêtre et non sur tous les équilibres.

D'où viennent `m` et `v` ? De la règle de Bayes pour deux gaussiennes : l'a priori `θ ~ N(0, τ²)` et le signal `s` se combinent en une croyance `θ | s ~ N(m, v)` dont les **précisions** s'additionnent (`1/v = 1/τ² + 1/σ²`) et dont la moyenne penche vers le signal à raison de sa précision relative. L'utilité de H en découle par intégration gaussienne — la probabilité que l'autre reçoive un signal au-dessus de `k` sachant le mien — et tout se résout en la forme close affichée : l'équilibre entier tient dans deux fonctions spéciales, `Φ` et `φ`.

In [4]:
# === Section 3 : le plus petit seuil d'équilibre symétrique ===
# A priori theta ~ N(0, TAU2).  Signal : s = theta + eps, eps ~ N(0, sigma^2).

def post(s, sigma):
    """Croyance a posteriori theta | s : (moyenne m, variance v)."""
    v = 1.0 / (1.0 / TAU2 + 1.0 / sigma**2)
    m = v * s / sigma**2
    return m, v

def utilite_H(s, k, sigma):
    """Utilité espérée de H pour un joueur de signal s, face a un seuil adverse k."""
    m, v = post(s, sigma)
    w = math.sqrt(v + sigma**2)
    z = (m - k) / w
    return (m + C) * Phi(z) + (v / w) * phi(z) - C

def g_sym(k, sigma):
    """Indifférence symétrique : utilite_H d'un joueur de signal s = k face a un
    adversaire de MÊME seuil k. Les racines de g_sym sont les équilibres de seuil."""
    return utilite_H(k, k, sigma)

def seuil_equilibre(sigma, k_max=30.0, n=2000):
    """Le plus PETIT seuil d'équilibre dans la fenêtre : première racine de g_sym,
    trouvée par balayage depuis k=0 puis bisection."""
    g0 = g_sym(0.0, sigma)
    if g0 >= 0.0:
        return 0.0
    ks = np.linspace(0.0, k_max, n)
    for i in range(n - 1):
        a, b = ks[i], ks[i + 1]
        ga = g_sym(a, sigma)
        if ga * g_sym(b, sigma) <= 0.0:          # racine dans [a, b]
            for _ in range(80):
                mid = 0.5 * (a + b)
                if ga * g_sym(mid, sigma) <= 0.0:
                    b = mid
                else:
                    a, ga = mid, g_sym(mid, sigma)
            return 0.5 * (a + b)
    return float("nan")

def compte_racines(sigma, k_max=None, n=40000):
    """Racines sur une fenêtre adaptée au bruit, avec résolution fixe.

    La seconde racine s'éloigne en 1/sigma : une borne fixe masque les
    petites valeurs de sigma. 30/sigma couvre les six cas mesurés.
    """
    if k_max is None:
        k_max = 30.0 / sigma
    step = k_max / n
    previous = g_sym(0.0, sigma)
    count = 0
    for i in range(1, n + 1):
        current = g_sym(i * step, sigma)
        if previous * current < 0:
            count += 1
        previous = current
    return count

sigma = 0.3
kstar = seuil_equilibre(sigma)
print(f"sigma = {sigma}   ->   plus petit seuil fini mesuré  k* = {kstar:.4f}")

sigma = 0.3   ->   plus petit seuil fini mesuré  k* = 1.5908


### Lecture — pourquoi k* dépasse C

À `σ = 0.3`, le plus petit seuil d'équilibre vaut `k* = 1.5908`, nettement au-dessus de `C = 1`. La lecture naïve — « le seuil devrait se caler sur le coût d'agir seul » — est fausse tant que le bruit est fini : l'écart `k* − C ≈ 0.59` mesure la **double incertitude** d'un joueur. Il doute d'abord sur le fondamental (il ne voit que `s`, pas `θ`), puis sur le signal de l'autre (même à fondamental donné, l'adversaire observe un `s'` différent du sien). Pour oser jouer H, il faut être optimiste sur `θ` **et** sur la probabilité que l'autre reçoive aussi un signal au-dessus du seuil — deux sources d'incertitude prises en compte par la même espérance conditionnelle, avec une racine mesurée au-dessus du cas sans bruit.

La forme close de l'indifférence symétrique encode ces croyances. Sur cette branche, la première racine est une traversée du négatif vers le positif ; une seconde traversée existe dans la fenêtre élargie. Le balayage localise la première racine et le comptage conserve les autres traversées au lieu de les confondre avec une sélection globale.


### Exercice 1

Complétez la fonction suivante ; les sondes restent ouvertes tant qu’elle renvoie `None`.


In [5]:
# Exercice 1 — Probabilité côté L
def prob_both_L(theta, sigmas):
    """Pour chaque sigma, proba ex-post que les DEUX jouent L (cas theta < 0).

    Attendu : une liste de même longueur que sigmas ; quand sigma -> 0 et theta < 0,
    chaque valeur doit tendre vers 1.
    """
    # TODO : compléter
    # Indice : symétrique de prob_both_H. P(un joueur joue L) = 1 - Phi((theta - k*)/sigma),
    #         avec k* = seuil_equilibre(sigma) ; les signaux sont indépendants.
    result = None  # TODO etudiant
    return result


sondes_ex1=[(-.5,[.3,.1,.01]),(1.2,[.3,.2]),(.8,[.1,.05])]
reponses_ex1=[prob_both_L(t,list(ss)) for t,ss in sondes_ex1]
if all(x is None for x in reponses_ex1): print("Exercice 1 à compléter")
else:
    for (t,ss),x in zip(sondes_ex1,reponses_ex1):
        assert x is not None and len(x)==len(ss), "Réponse partielle"
        for s,v in zip(ss,x):
            attendu=(1-Phi((t-seuil_equilibre(s))/s))**2
            assert np.isfinite(v) and abs(v-attendu)<1e-9, "Probabilité incorrecte"
    print("Exercice 1 : probabilités vérifiées")


Exercice 1 à compléter


In [6]:
# Multiplicité résiduelle : la borne de recherche suit le bruit.
print("Racines de g_sym sur [0, 30/sigma], avec 40000 intervalles :")
print(f"{'sigma':>7}  {'nb de seuils':>14}  {'plus petit k*':>15}")
for sigma in (0.3, 0.2, 0.1, 0.05, 0.01, 0.001):
    print(f"{sigma:>7.3f}  {compte_racines(sigma):>14}  {seuil_equilibre(sigma):>15.4f}")


Racines de g_sym sur [0, 30/sigma], avec 40000 intervalles :
  sigma    nb de seuils    plus petit k*
  0.300               2           1.5908
  0.200               2           1.2334


  0.100               2           1.0799


  0.050               2           1.0335


  0.010               2           1.0058


  0.001               2           1.0006


### Lecture — deux racines dans les fenêtres mesurées

Deux régularités traversent le tableau. D'abord le **nombre de racines : 2** à chaque niveau de bruit — la multiplicité n'est pas éliminée par le bruit, elle est réorganisée. Ensuite la **dérive du plus petit seuil** : `k*` descend de `1.5908` (σ = 0.3) à `1.0006` (σ = 0.001), et la cible de la convergence est `C = 1`, pas `0`. Quand l'observation devient presque parfaite, le seuil d'équilibre se fixe sur le **coût d'agir seul** — la frontière de dominance de risque de ce jeu symétrique complet : le mixte d'indifférence `p*=C/(θ+C)` vaut `1/2` en `θ=C`.

Le terme « robuste » est une appellation du carnet source pour la plus petite racine finie mesurée. Le balayage ne démontre ni résistance aux perturbations des croyances, ni sélection au sens de Carlsson et van Damme. Il exclut les équilibres hors de la classe de seuils et ne dit rien hors de la fenêtre.


### Exercice 2

Complétez la fonction suivante ; les sondes restent ouvertes tant qu’elle renvoie `None`.

Pour un bruit où la recherche ne trouve aucune racine finie dans `[0,30]`, conservez `NaN`.


In [7]:
# Exercice 2 — Le seuil se fixe sur le coût d'agir seul
def seuils_en_fonction_de_sigma(sigmas):
    """Renvoie k*(sigma) pour chaque sigma.

    Attendu : NaN signifie aucune racine finie trouvée sur la fenêtre ; on voit k* -> C quand sigma -> 0 (le seuil mesuré se rapproche
    sur le coût d'agir seul, pas sur 0).
    """
    # TODO : compléter
    # Indice : compréhension de liste — [seuil_equilibre(s) for s in sigmas]
    result = None  # TODO etudiant
    return result


sondes_ex2=[[1.,.3,.05,.001],[.2,.1]]
reponses_ex2=[seuils_en_fonction_de_sigma(list(ss)) for ss in sondes_ex2]
if all(x is None for x in reponses_ex2): print("Exercice 2 à compléter")
else:
    for ss,x in zip(sondes_ex2,reponses_ex2):
        assert x is not None and len(x)==len(ss), "Réponse partielle"
        for s,v in zip(ss,x):
            attendu=seuil_equilibre(s)
            assert (np.isnan(v) if np.isnan(attendu) else np.isfinite(v) and abs(v-attendu)<1e-9), "Seuil incorrect"
    print("Exercice 2 : seuils vérifiés")


Exercice 2 à compléter


In [8]:
# Probabilité (interne) de jouer H à l'équilibre : la part de joueurs qui jouent H.
#   s ~ N(theta, sigma^2)   =>   P(H) = P(s >= k*) = Phi((theta - k*) / sigma).
theta, sigma = 1.2, 0.3
kstar = seuil_equilibre(sigma)
p_H = Phi((theta - kstar) / sigma)
print(f"theta = {theta},  sigma = {sigma},  k* = {kstar:.4f}")
print(f"P(jouer H à l'équilibre) = {p_H:.4f}   ->  interne : le plus petit seuil fixe une part de population.")

theta = 1.2,  sigma = 0.3,  k* = 1.5908
P(jouer H à l'équilibre) = 0.0964   ->  interne : le plus petit seuil fixe une part de population.


### Probabilités sur la plus petite racine finie

Pour le plus petit seuil trouvé dans la fenêtre, la part conditionnelle de joueurs qui jouent H est `P(H) = Φ((θ − k*)/σ)` — une valeur **interne** (ni 0 ni 1). D'autres racines peuvent exister ; cette mesure ne prouve pas leur sélection ou leur rejet.

**Branche bornée à petit bruit.** Pour `k` borné, la formule donne `m→k`, `v/w→0` et `(m-k)/w→0`, donc `g_sym(k,σ)→(k-C)/2`. Une branche de racines qui admet une limite finie doit donc tendre vers `C` ; cela ne décrit pas les racines qui s'échappent de la fenêtre. Sur cette branche, les signaux convergent vers `θ`, donc :

- si **`θ > C`** : `P(les deux jouent H) → 1`  →  la probabilité de (H,H) tend vers 1 sur cette branche ;
- si **`θ < C`** : `P(les deux jouent H) → 0`  →  la probabilité de (H,H) tend vers 0 sur cette branche — et c'est un **échec de coordination** dès que `0 < θ < C` (l'équilibre efficace (H,H) existait, cette branche n'y conduit pas).

Dans les valeurs calculées, la transition de cette branche se rapproche de `C`. La fenêtre et la classe de seuils ne suffisent pas à démontrer une sélection robuste par perturbations.

La borne `θ = C` est un énoncé de limite. À bruit fini, les probabilités ne sont pas encore extrêmes : les colonnes de la cellule suivante montrent le phénomène en train de se faire, du « presque jamais » à bruit fort au « presque toujours » à bruit faible — et la transition entre les deux est raide. C'est cette raideur, mesurée, qui fait des jeux globaux un instrument prédictif : la frontière est fine et calculable.

In [9]:
# === Probabilités associées au plus petit seuil fini ===
#   P(les deux jouent H) = [ Phi((theta - k*(sigma)) / sigma) ]^2   (signaux indépendants).

def prob_both_H(theta, sigma):
    k = seuil_equilibre(sigma)
    p = Phi((theta - k) / sigma)
    return p * p

sigmas = [0.3, 0.2, 0.1, 0.05, 0.01, 0.001]

print("theta = +1.2   (theta > C=1  ->  probabilité de (H,H) sur la branche mesurée)")
print(f"{'sigma':>7}  {'k*':>9}  {'P(les 2 jouent H)':>18}")
for s in sigmas:
    print(f"{s:>7.3f}  {seuil_equilibre(s):>9.4f}  {prob_both_H(1.2, s):>18.4f}")

print("\ntheta = +0.8   (theta < C=1  ->  probabilité de (H,H) sur la branche mesurée)")
print(f"{'sigma':>7}  {'k*':>9}  {'P(les 2 jouent H)':>18}")
for s in sigmas:
    print(f"{s:>7.3f}  {seuil_equilibre(s):>9.4f}  {prob_both_H(0.8, s):>18.4f}")

theta = +1.2   (theta > C=1  ->  probabilité de (H,H) sur la branche mesurée)
  sigma         k*   P(les 2 jouent H)
  0.300     1.5908              0.0093
  0.200     1.2334              0.1880
  0.100     1.0799              0.7836
  0.050     1.0335              0.9991
  0.010     1.0058              1.0000
  0.001     1.0006              1.0000

theta = +0.8   (theta < C=1  ->  probabilité de (H,H) sur la branche mesurée)
  sigma         k*   P(les 2 jouent H)
  0.300     1.5908              0.0000
  0.200     1.2334              0.0002
  0.100     1.0799              0.0000
  0.050     1.0335              0.0000
  0.010     1.0058              0.0000
  0.001     1.0006              0.0000


### Lecture — la transition sur la branche mesurée

Pour `θ = 1.2` (au-dessus de `C`), la coordination sur (H, H) grimpe de `0.0093` à `σ = 0.3` — le seuil `k* = 1.5908` reste hors de portée de la plupart des signaux — puis perce : `0.1880` à `σ = 0.2`, `0.7836` à `σ = 0.1`, `0.9991` à `σ = 0.05`, et `1.0000` dès `σ = 0.01`. Pour `θ = 0.8` (sous `C`), la colonne reste collée à zéro, `0.0002` au maximum à `σ = 0.2` : sur un projet dont la rentabilité ne couvre pas le coût d'agir seul, la coordination sur (H,H) reste très peu probable pour les bruits de la table.

La transition de la première colonne est raide : entre `σ = 0.2` et `σ = 0.05`, un facteur 4 de bruit fait passer la probabilité d'un cinquième à la quasi-certitude. C'est le **mur** — la transition est raide. Et son point de bascule n'est pas `θ = 0` (apparition de l'accord pur (H,H) dans le jeu complet) mais `θ = C`. Entre les deux s'ouvre la zone des échecs de coordination : `0 < θ < C` rendrait (H, H) efficient à information complète, et la branche de seuils mesurée rend (H,H) peu probable au profit de (L,L) — la crise de change qui n'a pas lieu parce que chacun doute que l'autre attaque.
Ces probabilités portent sur la plus petite racine finie trouvée dans la fenêtre, et ne prouvent pas une sélection par perturbations ni l’absence d’équilibres hors classe.


### Exercice 3

Complétez la fonction suivante ; les sondes restent ouvertes tant qu’elle renvoie `None`.


In [10]:
# Exercice 3 — Une seule racine dans une fenêtre donnée ?
def est_unique(sigma, k_max=None):
    """True si g_sym a exactement une racine dans la fenêtre mesurée.

    Sans k_max explicite, utiliser la même fenêtre 30/sigma que la cellule
    de mesure : le résultat porte sur cette fenêtre, pas sur [0, infini).
    """
    # TODO : compléter
    # Indice : appeler compte_racines(sigma, k_max) et comparer à 1.
    result = None  # TODO etudiant
    return result


sondes_ex3=[(.05,None),(.3,3.),(.3,1.)]
reponses_ex3=[est_unique(s,k) for s,k in sondes_ex3]
if all(x is None for x in reponses_ex3): print("Exercice 3 à compléter")
else:
    for (s,k),x in zip(sondes_ex3,reponses_ex3):
        assert isinstance(x,(bool,np.bool_)) and x == (compte_racines(s,k)==1), "Comptage incorrect ou réponse partielle"
    print("Exercice 3 : fenêtres vérifiées")


Exercice 3 à compléter


## 4. Synthèse

| Information | Équilibres étudiés | Résultat quand `σ → 0` |
|---|---|---|
| **Complète** (`θ` connu) | **multiplicité** : (H,H), (L,L), mixte | — |
| **Incomplète** (signal `s = θ + ε`) | **plus petit seuil fini mesuré** `k*` ; autres racines possibles | probabilité de (H,H) sur la branche mesurée tendant vers 1 si `θ > C`, vers 0 si `θ < C` |

La fenêtre et le calcul à seuil donnent une branche mesurée, sans démontrer une sélection par perturbations. C'est ce qui rend les jeux globaux utiles pour étudier les crises de change, les paniques bancaires et la coordination sur un fondamental.

Les références héritées du Python citent Morris & Shin (1998) comme application aux attaques de change. Les gains H/H et H/L de ce carnet en donnent une analogie pédagogique : une action collective peut réussir alors qu'une action isolée coûte C. Le calcul présent fait varier la précision des **signaux privés** ; il ne mesure ni une communication publique ni l'effet d'une intervention de banque centrale. Les probabilités rapportées portent sur la branche de seuils définie ci-dessus.


### Synthèse des exercices

Les trois exercices réutilisent la probabilité de coordination, les seuils et le comptage des racines dans une fenêtre. Les contrôles portent sur les données mesurées et ne démontrent pas une sélection au-delà de ces fenêtres.


## Références

- **Carlsson, H., & van Damme, E. (1993).** « Global Games and Equilibrium Selection », *Econometrica* 61(5), 989–1018. — introduction du jeu global et de la sélection d'équilibre sous information bruitée.
- **Morris, S., & Shin, H. S. (1998).** « Unique Equilibrium in a Model of Self-Fulfilling Currency Attacks », *American Economic Review* 88(3), 587–597. — application canonique aux attaques de change.

**Place dans la série** : `11e`, « Autour de 11 » — avec `11` (jeux bayésiens), `11b` (Lean), `11c` (trembles) et `11d` (raffinements séquentiels).
